In [1]:
import matplotlib.style as mplstyle
mplstyle.use('fast')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
from load_results import load, format
from scipy.stats import chi2 as chi2_dist
import itertools

from schedule_similarity import SCRIPTS, KS, NORMALIZE
from schedule_representations import experiments


In [ ]:

df = load()
# test for completeness
SCRIPT_META = {
    s: {
        "module": s.replace(".py", "").replace("/", "."),
        "label": s.split("/")[1].replace("run_", "").replace(".py", "")
    }
    for s in SCRIPTS
}

if_sorting_strategies = [
    "most_harmful",
    "most_helpful",
    "least_influential",
    "most_influential",
]


df_pre = df.copy()

group_keys = [
    "Experiment",
    "model",
    "test_dataset",
    "test_dataset_split",
    "train_dataset",
    "train_dataset_split",
    "Rep_A",
    "Rep_B",
    "k",
    "normalize",
]

df_grouped = df_pre.groupby(group_keys, sort=False)


for exp in experiments:
    for script_a, script_b in itertools.combinations_with_replacement(SCRIPTS, 2):

        meta_a = SCRIPT_META[script_a]
        meta_b = SCRIPT_META[script_b]

        for normalize in NORMALIZE:
            for k in KS:

                base_key = (
                    exp["name"],
                    exp["model"],
                    exp["dataset"],
                    "test",
                    exp["dataset"],
                    "test",
                    script_a.replace("/", ".").replace(".py", ""),
                    script_b.replace("/", ".").replace(".py", ""),
                    k,
                    normalize,
                )
                
                if (base_key not in df_grouped.groups):
                    print(base_key)
                    raise NotImplementedError

                df_subset_base = df_grouped.get_group(base_key)

                for if_sorting_strategy_a in if_sorting_strategies:
                    for if_sorting_strategy_b in if_sorting_strategies:
                        df_subset = df_subset_base[
                            (
                                (df_subset_base["selection_strategy_name_a"] == if_sorting_strategy_a)
                                | (df_subset_base["selection_strategy_name_a"] == "-")
                            )
                            &
                            (
                                (df_subset_base["selection_strategy_name_b"] == if_sorting_strategy_b)
                                | (df_subset_base["selection_strategy_name_b"] == "-")
                            )
                        ].dropna() # so that asserts fail on nan


                        if normalize:
                            if k == -1:
                                assert len(df_subset) == 2, display(df_subset)
                            else:
                                assert len(df_subset) == 4, display(df_subset)
                        else:
                            if k == -1:
                                assert len(df_subset) == 6, display(df_subset)
                            else:
                                assert len(df_subset) == 10, display(df_subset)

KeyError: ('ftrace', './models/Qwen3-1.7B-Base_ftrace_date-lm_conversational_lr0.0001_seed42', 'loris3/ftrace_date-lm_conversational-sample', 'test', 'loris3/ftrace_date-lm_conversational-sample', 'test', 'representations.run_gradsim_normalized', 'representations.run_gradsim_normalized', -1, True)

In [ ]:
df = format(df)


In [ ]:
df.columns

Index(['Experiment', 'Rep_A', 'Rep_B', 'score_name', 'score', 'k', 'normalize',
       'model', 'train_dataset', 'train_dataset_split', 'test_dataset',
       'test_dataset_split', 'base_model', 'test_type'],
      dtype='object')

In [ ]:
assert len(df[df["score"].isna()]) == 0

In [ ]:
df

,Experiment,Rep_A,Rep_B,score_name,score,k,normalize,model,train_dataset,train_dataset_split,test_dataset,test_dataset_split,base_model,test_type
0,code,$r^{L2}_{\nabla f} {most inf}$,$r_{{bm25}}$,sign_agreement,1.000000,-1,False,./models/Llama-3.2-1B_tulu-3-sft-personas-code...,loris3/tulu-3-sft-personas-code-sample,test,loris3/tulu-3-sft-personas-code-sample,test,Llama-3.2-1B,func
1,code,$r^{L2}_{\nabla f} {most inf}$,$r_{{bm25}}$,rbo,0.504441,-1,False,./models/Llama-3.2-1B_tulu-3-sft-personas-code...,loris3/tulu-3-sft-personas-code-sample,test,loris3/tulu-3-sft-personas-code-sample,test,Llama-3.2-1B,func
2,code,$r^{L2}_{\nabla f} {most inf}$,qwen,spearman_rho,0.023215,-1,False,./models/Llama-3.2-1B_tulu-3-sft-personas-code...,loris3/tulu-3-sft-personas-code-sample,test,loris3/tulu-3-sft-personas-code-sample,test,Llama-3.2-1B,func
3,code,$r^{L2}_{\nabla f} {most inf}$,qwen,ndcg,0.687478,-1,False,./models/Llama-3.2-1B_tulu-3-sft-personas-code...,loris3/tulu-3-sft-personas-code-sample,test,loris3/tulu-3-sft-personas-code-sample,test,Llama-3.2-1B,func
4,code,$r^{L2}_{\nabla f} {most inf}$,qwen,sign_agreement,1.000000,-1,False,./models/Llama-3.2-1B_tulu-3-sft-personas-code...,loris3/tulu-3-sft-personas-code-sample,test,loris3/tulu-3-sft-personas-code-sample,test,Llama-3.2-1B,func
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44533,code,$r^{L2}_{\nabla f} {least inf}$,$r_{{bm25}}$,ndcg,0.994595,-1,False,./models/Llama-3.2-1B_tulu-3-sft-personas-code...,loris3/tulu-3-sft-personas-code-sample,test,loris3/tulu-3-sft-personas-code-sample,test,Llama-3.2-1B,func
44534,code,$r^{L2}_{\nabla f} {least inf}$,$r_{{bm25}}$,sign_agreement,0.000000,-1,False,./models/Llama-3.2-1B_tulu-3-sft-personas-code...,loris3/tulu-3-sft-personas-code-sample,test,loris3/tulu-3-sft-personas-code-sample,test,Llama-3.2-1B,func
44535,code,$r^{L2}_{\nabla f} {least inf}$,$r_{{bm25}}$,rbo,0.497797,-1,False,./models/Llama-3.2-1B_tulu-3-sft-personas-code...,loris3/tulu-3-sft-personas-code-sample,test,loris3/tulu-3-sft-personas-code-sample,test,Llama-3.2-1B,func
44536,code,$r^{L2}_{\nabla f} {most inf}$,$r_{{bm25}}$,spearman_rho,0.016792,-1,False,./models/Llama-3.2-1B_tulu-3-sft-personas-code...,loris3/tulu-3-sft-personas-code-sample,test,loris3/tulu-3-sft-personas-code-sample,test,Llama-3.2-1B,func


In [ ]:
long_df = df[["score_name", "base_model", "Experiment", "train_dataset", "Rep_A", "Rep_B", "test_type", "k", "normalize", "score"]].copy().rename(columns={"score_name": "metric", "train_dataset": "dataset"})
long_df["Rep_pair"] = long_df.apply(
    lambda r: ":".join(sorted([str(r["Rep_A"]), str(r["Rep_B"])])),
    axis=1
)
long_df = long_df.drop(columns=["Rep_A", "Rep_B"])
assert len(long_df[long_df.isna().any(axis=1)]) == 0
long_df

,metric,base_model,Experiment,dataset,test_type,k,normalize,score,Rep_pair
0,sign_agreement,Llama-3.2-1B,code,loris3/tulu-3-sft-personas-code-sample,func,-1,False,1.000000,$r^{L2}_{\nabla f} {most inf}$:$r_{{bm25}}$
1,rbo,Llama-3.2-1B,code,loris3/tulu-3-sft-personas-code-sample,func,-1,False,0.504441,$r^{L2}_{\nabla f} {most inf}$:$r_{{bm25}}$
2,spearman_rho,Llama-3.2-1B,code,loris3/tulu-3-sft-personas-code-sample,func,-1,False,0.023215,$r^{L2}_{\nabla f} {most inf}$:qwen
3,ndcg,Llama-3.2-1B,code,loris3/tulu-3-sft-personas-code-sample,func,-1,False,0.687478,$r^{L2}_{\nabla f} {most inf}$:qwen
4,sign_agreement,Llama-3.2-1B,code,loris3/tulu-3-sft-personas-code-sample,func,-1,False,1.000000,$r^{L2}_{\nabla f} {most inf}$:qwen
...,...,...,...,...,...,...,...,...,...
44533,ndcg,Llama-3.2-1B,code,loris3/tulu-3-sft-personas-code-sample,func,-1,False,0.994595,$r^{L2}_{\nabla f} {least inf}$:$r_{{bm25}}$
44534,sign_agreement,Llama-3.2-1B,code,loris3/tulu-3-sft-personas-code-sample,func,-1,False,0.000000,$r^{L2}_{\nabla f} {least inf}$:$r_{{bm25}}$
44535,rbo,Llama-3.2-1B,code,loris3/tulu-3-sft-personas-code-sample,func,-1,False,0.497797,$r^{L2}_{\nabla f} {least inf}$:$r_{{bm25}}$
44536,spearman_rho,Llama-3.2-1B,code,loris3/tulu-3-sft-personas-code-sample,func,-1,False,0.016792,$r^{L2}_{\nabla f} {most inf}$:$r_{{bm25}}$


$y_{i,m,d}=\beta_0+\beta^{(m)}_{m}+\beta^{(d)}_{d}+\beta^{(m \times d)}_{m,d}+u_i+\epsilon_{i,m,d}$

In [ ]:
def fit_model(definition, long_df, score_name, k, normalize ):
    if normalize:
        return None
    df = long_df[(long_df["metric"] == score_name) & (long_df["k"] == k)  & (long_df["normalize"] == normalize)].copy()
    
    cols = ["base_model", "dataset", "score", "Rep_pair"]
    nan_rows = df[df[cols].isnull().any(axis=1)]
    
    if df[cols].isnull().values.any():
        print(f"missing entries for {score_name} at k={k}")
        display(nan_rows)
        print(len(df),len(nan_rows))
        return None
    if (df[cols].nunique() == 1).any():
        print(f"constant entries for {score_name} at k={k}")
        return None
    data = df[cols].reset_index(drop=True)
    # display(data)
    # print(data.groupby("Rep_pair").size().describe())
    return smf.mixedlm(
                definition,
                data=data,
                groups=data["Rep_pair"]
            ).fit()

configs = long_df.groupby(["metric", "k", "test_type", "normalize"]).groups.keys()
models = {
    (score_name, k, test_type, normalize) : fit_model("score ~ C(base_model) * C(dataset)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize in configs
} 

models_no_interaction = {
    (score_name, k, test_type, normalize) : fit_model("score ~ C(base_model) + C(dataset)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize  in configs
} 

models_no_base_model = {
    (score_name, k, test_type, normalize) : fit_model("score ~ C(dataset)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize  in configs
} 

models_no_dataset = {
    (score_name, k, test_type, normalize) : fit_model("score ~ C(base_model)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize  in configs
} 


/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


$H_0: \beta^{(m \times d)}_{m,d} = 0 \quad \forall m,d,$



$H_0^{(M)}: \beta^{(m)}_{m} = 0 \quad \forall m$

$H_0^{(D)}: \beta^{(d)}_{d} = 0 \quad \forall d$

In [ ]:


def lr_test_df(full_model, reduced_model, term_name):
    lr_stat = 2 * (full_model.llf - reduced_model.llf)
    df_diff = full_model.df_modelwc - reduced_model.df_modelwc
    p_value = chi2_dist.sf(lr_stat, df_diff)

    return pd.DataFrame({
        term_name: [term_name],
        "chi2": [lr_stat],
        "P>chi2": [p_value],
        "df constraint": [df_diff]
    }).set_index(term_name)
def compute_r2_mixedlm(model):
    # fixed effects design matrix
    fe = model.model.exog
    fe_params = model.fe_params.values

    # fixed-effect linear predictor η
    eta = fe @ fe_params
    sigma_f2 = np.var(eta, ddof=1)

    # random intercept variance (Rep_pair)
    sigma_pair2 = model.cov_re.values.item()

    # residual variance
    sigma_e2 = model.scale

    denom = sigma_f2 + sigma_pair2 + sigma_e2

    R2_m = sigma_f2 / denom
    R2_c = (sigma_f2 + sigma_pair2) / denom

    return sigma_f2, sigma_pair2, sigma_e2, R2_m, R2_c


result_dfs_lrt = []
result_dfs_r2 = []
for (score_name, k, test_type, normalize), model in models.items():
    if model is None:
        continue
    interaction_test = pd.concat([
        lr_test_df(
            model,
            models_no_base_model[(score_name, k, test_type, normalize)],
            "C(base_model)"
        ),
        lr_test_df(
            model,
            models_no_dataset[(score_name, k, test_type,normalize)],
            "C(dataset)"
        ),
        lr_test_df(
            model,
            models_no_interaction[(score_name, k, test_type,normalize)],
            "C(base_model):C(dataset)"
        )
        ])
    # display(interaction_test)
    df_model_dataset = interaction_test.stack().rename_axis(["term", "stat"]).to_frame().T.reset_index(drop=True)
    df_model_dataset["metric"] = score_name
    df_model_dataset["test_type"] = test_type
    df_model_dataset["k"] = k
    df_model_dataset["normalize"] = normalize
    df_model_dataset = df_model_dataset.set_index(["metric", "k", "normalize"])
    result_dfs_lrt.append(df_model_dataset)
    
    r2_dfs = []
    for setting, model_ in [
        ("full", model),
        ("no_interaction", models_no_interaction[(score_name, k,test_type, normalize)]),
        ("no_base_model", models_no_base_model[(score_name, k,test_type,normalize)]),
        ("no_dataset", models_no_dataset[(score_name, k,test_type,normalize)])
        ]:
        sigma_f2, sigma_pair2, sigma_e2, R2_m, R2_c = compute_r2_mixedlm(model_)
        df_r2 = pd.DataFrame({

            "metric": [score_name],
            "test_type": [test_type],
            "normalize": [normalize],
            "k": [k],
            # "sigma_f2": [sigma_f2],
            # "sigma_pair2": [sigma_pair2],
            # "sigma_e2": [sigma_e2],
            "R2_m": [R2_m],
            "R2_c": [R2_c]
        })
        df_r2.columns = pd.MultiIndex.from_product(
            [[setting], df_r2.columns]
        )

        r2_dfs.append(df_r2)
    result_dfs_r2.append(pd.concat(r2_dfs, axis=1))
        
    

    
 
df_lrt = pd.concat(result_dfs_lrt)
df_r2_model_dataset = pd.concat(result_dfs_r2)




In [ ]:
df_display = df_lrt.reset_index().set_index(["test_type", "metric", "k"]).sort_values(["test_type", "metric", "k"])
df_display = df_display.loc[:, df_display.columns.get_level_values("stat") != "df constraint"]

chi2_cols = df_display.columns.get_level_values("stat") == "chi2"


df_display.style.background_gradient(
    cmap="viridis",
    subset=df_display.columns[chi2_cols],
    axis=1
)

In [ ]:
df = df_r2_model_dataset.copy()

df = df.set_index(("full", "metric"))

delta_r2 = pd.DataFrame(index=df.index)

delta_r2["interaction_model_dataset_R2_m"] = (
    df[("full", "R2_m")] - df[("no_interaction", "R2_m")]
)
delta_r2["interaction_model_dataset_R2_c"] = (
    df[("full", "R2_c")] - df[("no_interaction", "R2_c")]
)

delta_r2["base_model_R2_m"] = (
    df[("full", "R2_m")] - df[("no_base_model", "R2_m")]
)
delta_r2["base_model_R2_c"] = (
    df[("full", "R2_c")] - df[("no_base_model", "R2_c")]
)

delta_r2["dataset_R2_m"] = (
    df[("full", "R2_m")] - df[("no_dataset", "R2_m")]
)
delta_r2["dataset_R2_c"] = (
    df[("full", "R2_c")] - df[("no_dataset", "R2_c")]
)



r2_m_cols = [col for col in delta_r2.columns if "R2_m" in col]


delta_r2.style.background_gradient(
    cmap="viridis",
    subset=r2_m_cols,
    axis=1
)

,interaction_model_dataset_R2_m,interaction_model_dataset_R2_c,base_model_R2_m,base_model_R2_c,dataset_R2_m,dataset_R2_c
"('full', 'metric')",,,,,,
cka,0.002191,0.000478,0.009715,0.007763,0.006766,0.004887
ndcg,0.001756,0.001528,0.002533,0.002255,0.021532,0.021780
rbo,0.000229,-0.000384,0.000610,-0.000148,0.002131,0.001468
sign_agreement,0.000001,-0.000069,0.000001,-0.000086,0.000067,-0.000012
spearman_rho,0.000169,-0.000482,0.000693,-0.000107,0.001476,0.000754
unbiased_cka,0.001597,0.000174,0.003841,0.002113,0.003162,0.001541


### 